# Breaking Bad — Decompress Artifact Subset (10/object) trên Kaggle

**Mục tiêu:** decompress 10 fracture ngẫu nhiên/object (5 mode + 5 fractured) cho subset `artifact`, output sẵn sàng dùng cho training DiffusionNet.

**Chiến lược**: sub-sample TRƯỚC khi decompress để tránh vượt giới hạn 20GB của Kaggle output.

**Kết quả**: folder `artifact/` chứa mesh `.obj` thật, mỗi object có ~10 fractures, mỗi fracture có nhiều `piece_X.obj`. Tổng ~3–5 GB.

## Yêu cầu trước khi chạy

1. **Upload 3 file lên Kaggle Dataset** (gọi tên ví dụ `breaking-bad-artifact-raw`):
   - `artifact_compressed.zip` (189 MB)
   - `volume_constrained-artifact_compressed.zip` (197 MB) — *optional, dùng nếu muốn version vỡ đều*
   - `data_split.tar.gz` (26 KB)

2. **Notebook settings (panel phải)**:
   - Accelerator: CPU (decompress không cần GPU)
   - Internet: **ON**
   - Persistence: **Files only**
   - Attach dataset `breaking-bad-artifact-raw` qua **Add Input**

3. Sau khi notebook chạy xong, dùng **Save Version** để snapshot output rồi tạo Kaggle Dataset mới từ `/kaggle/working/artifact/` để các notebook training sau dùng được.

Setup cho subset Breaking Bad Artifact

## 1. Cấu hình

In [ ]:
import os, sys, shutil, random, json
from pathlib import Path

# === CONFIG ===
SEED = 42
N_MODES_PER_OBJECT = 5      # số mode_X giữ lại
N_FRACTURED_PER_OBJECT = 5  # số fractured_X giữ lại
USE_VOLUME_CONSTRAINED = False  # True nếu muốn dùng phiên bản vỡ đều

# === PATHS ===
INPUT_DIR = '/kaggle/input/breaking-bad-artifact-raw'  # ⚠️ ĐỔI theo tên dataset bạn tạo
WORK_DIR = '/kaggle/working'
DATA_ROOT = f'{WORK_DIR}/bb_data'           # nơi chứa compressed sau khi extract
FILTERED_ROOT = f'{WORK_DIR}/bb_filtered'   # compressed đã sub-sample
OUTPUT_ROOT = f'{WORK_DIR}/artifact'        # output cuối — meshes thật

random.seed(SEED)
print(f'Sẽ giữ {N_MODES_PER_OBJECT} mode + {N_FRACTURED_PER_OBJECT} fractured per object')
print(f'Source: {"volume_constrained" if USE_VOLUME_CONSTRAINED else "regular"} artifact subset')

## 2. Giải nén dataset gốc

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)

# Chọn zip artifact phù hợp
if USE_VOLUME_CONSTRAINED:
    artifact_zip = f'{INPUT_DIR}/volume_constrained-artifact_compressed.zip'
else:
    artifact_zip = f'{INPUT_DIR}/artifact_compressed.zip'

print(f'Đang giải nén {artifact_zip}...')
!unzip -q '{artifact_zip}' -d {DATA_ROOT}/

print('Đang giải nén data_split.tar.gz...')
!tar -xzf {INPUT_DIR}/data_split.tar.gz -C {DATA_ROOT}/

print('\nCấu trúc:')
!ls -la {DATA_ROOT}/
print()
print('Số object trong artifact_compressed:')
!ls {DATA_ROOT}/artifact_compressed/ | wc -l

## 3. Sub-sample: chỉ giữ 10 fracture/object

Với mỗi object, random pick:
- 5 `mode_X` (từ 20 mode có sẵn)
- 5 `fractured_X` (từ 80 fractured có sẵn)

Tạo copy gọn vào `bb_filtered/` để chạy decompress.

In [ ]:
src = Path(f'{DATA_ROOT}/artifact_compressed')
dst = Path(FILTERED_ROOT) / 'artifact_compressed'
dst.mkdir(parents=True, exist_ok=True)

objects = sorted([d for d in src.iterdir() if d.is_dir()])
print(f'Tổng số object: {len(objects)}')

kept_summary = []

for obj_dir in objects:
    obj_dst = dst / obj_dir.name
    obj_dst.mkdir(exist_ok=True)
    
    # Copy 2 file gốc bắt buộc
    for required in ['compressed_mesh.obj', 'compressed_data.npz']:
        src_file = obj_dir / required
        if src_file.exists():
            shutil.copy(src_file, obj_dst / required)
    
    # Liệt kê các mode_X và fractured_X
    modes = sorted([d for d in obj_dir.iterdir() if d.is_dir() and d.name.startswith('mode_')])
    fractured = sorted([d for d in obj_dir.iterdir() if d.is_dir() and d.name.startswith('fractured_')])
    
    # Pick random
    picked_modes = random.sample(modes, min(N_MODES_PER_OBJECT, len(modes)))
    picked_fractured = random.sample(fractured, min(N_FRACTURED_PER_OBJECT, len(fractured)))
    
    # Copy folder
    for d in picked_modes + picked_fractured:
        shutil.copytree(d, obj_dst / d.name)
    
    kept_summary.append({
        'object': obj_dir.name,
        'n_modes_total': len(modes),
        'n_fractured_total': len(fractured),
        'kept_modes': [m.name for m in picked_modes],
        'kept_fractured': [f.name for f in picked_fractured],
    })

# Save manifest để biết đã chọn gì (reproducibility)
with open(f'{WORK_DIR}/sample_manifest.json', 'w') as f:
    json.dump({'seed': SEED, 'objects': kept_summary}, f, indent=2)

print(f'Đã sub-sample {len(kept_summary)} object.')
print(f'Manifest lưu ở: {WORK_DIR}/sample_manifest.json')

# Xóa folder gốc để giải phóng disk
shutil.rmtree(DATA_ROOT + '/artifact_compressed')
print('Đã xóa folder gốc để tiết kiệm disk.')

# Move data_split sang filtered root
shutil.move(f'{DATA_ROOT}/data_split', f'{FILTERED_ROOT}/data_split')
print('Đã chuyển data_split vào filtered root.')

print()
print('Disk usage:')
!du -sh {FILTERED_ROOT}/*

## 4. Clone repo decompress + cài dependencies

In [ ]:
REPO_DIR = f'{WORK_DIR}/bb-decompress'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Breaking-Bad-Dataset/Breaking-Bad-Dataset.github.io.git {REPO_DIR}

print('Files trong repo:')
!ls {REPO_DIR}/

In [ ]:
# Cài dependencies cho decompress
# `libigl` và `gpytoolbox` là 2 dependency khó nhất
!pip install -q numpy scipy tqdm
!pip install -q libigl
!pip install -q gpytoolbox==0.2.0

# Verify
import igl
import gpytoolbox
print(f'libigl: {igl.__version__ if hasattr(igl, "__version__") else "installed"}')
print(f'gpytoolbox: {gpytoolbox.__version__ if hasattr(gpytoolbox, "__version__") else "installed"}')

## 5. Chạy decompress

Script `decompress.py` sẽ đọc compressed format và xuất ra `piece_X.obj` cho mỗi mảnh vỡ.

In [ ]:
# Chạy decompress
import subprocess

cmd = [
    'python', f'{REPO_DIR}/decompress.py',
    '--data_root', FILTERED_ROOT,
    '--subset', 'artifact',
]

print(f'Đang chạy: {" ".join(cmd)}')
print('(Có thể mất 20-40 phút cho ~2000 fractures...)')
print()

result = subprocess.run(cmd, capture_output=False, cwd=REPO_DIR)
print(f'\nReturn code: {result.returncode}')

## 6. Kiểm tra output

In [ ]:
# Output mong đợi: {FILTERED_ROOT}/artifact/{object_id}_sf/{mode|fractured}_X/piece_X.obj
decompressed_root = Path(FILTERED_ROOT) / 'artifact'

if not decompressed_root.exists():
    print(f'❌ Không thấy {decompressed_root}. Kiểm tra log decompress.')
else:
    objects = sorted([d for d in decompressed_root.iterdir() if d.is_dir()])
    print(f'✅ Số object decompressed: {len(objects)}')
    
    # Sample 1 object
    if objects:
        sample = objects[0]
        print(f'\nSample object: {sample.name}')
        sub = sorted([d for d in sample.iterdir() if d.is_dir()])
        print(f'  Số fracture: {len(sub)}')
        if sub:
            frac = sub[0]
            pieces = sorted(frac.glob('piece_*.obj'))
            print(f'  Sample fracture {frac.name}: {len(pieces)} piece(s)')
            if pieces:
                print(f'  File mẫu: {pieces[0].name} — {pieces[0].stat().st_size} bytes')
    
    print(f'\nDisk usage:')
    !du -sh {decompressed_root}

## 7. Visualize 1 mẫu — xác nhận mesh OK

In [ ]:
!pip install -q trimesh plotly

import trimesh
import numpy as np
import plotly.graph_objects as go

# Lấy 1 fracture mẫu
sample_obj = sorted(decompressed_root.iterdir())[0]
sample_frac = sorted([d for d in sample_obj.iterdir() if d.is_dir()])[0]
pieces_files = sorted(sample_frac.glob('piece_*.obj'))

print(f'Visualize object={sample_obj.name}, fracture={sample_frac.name}, {len(pieces_files)} piece(s)')

colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow', 'plum', 'lightsalmon', 'cyan', 'gold']

traces = []
for i, p_file in enumerate(pieces_files):
    m = trimesh.load(str(p_file))
    v = np.asarray(m.vertices)
    f = np.asarray(m.faces)
    traces.append(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=f[:,0], j=f[:,1], k=f[:,2],
        color=colors[i % len(colors)],
        opacity=0.9, name=p_file.stem,
        flatshading=True,
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    title=f'{sample_obj.name} / {sample_frac.name}',
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

## 8. Tạo cấu trúc output cuối + cleanup

Move `artifact/` và `data_split/` lên `/kaggle/working/` để Save Version đóng gói dễ.

In [ ]:
# Move output lên working root để Save Version
if not Path(OUTPUT_ROOT).exists():
    shutil.move(str(decompressed_root), OUTPUT_ROOT)

data_split_final = f'{WORK_DIR}/data_split'
if not Path(data_split_final).exists():
    shutil.move(f'{FILTERED_ROOT}/data_split', data_split_final)

# Move manifest
manifest_final = f'{WORK_DIR}/sample_manifest.json'

# Cleanup: xóa folder filtered (đã không cần)
if Path(FILTERED_ROOT).exists():
    shutil.rmtree(FILTERED_ROOT)

# Cleanup: xóa folder repo decompress
if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print('Cấu trúc cuối ở /kaggle/working/:')
!ls -la {WORK_DIR}/
print()
print('Tổng disk usage:')
!du -sh {WORK_DIR}/*

## 9. Tiếp theo: Save Version + Tạo Kaggle Dataset

1. Bấm **Save Version** (top right) → chọn **Save & Run All** → đợi notebook chạy lại từ đầu và snapshot output.
2. Sau khi version hoàn thành, vào **Output** tab của notebook → chọn **Create Dataset from Output** → đặt tên ví dụ `breaking-bad-artifact-decompressed`.
3. Trong notebook training tiếp theo, attach dataset này qua **Add Input**. Files sẽ ở `/kaggle/input/breaking-bad-artifact-decompressed/artifact/...`

Như vậy bạn không cần decompress lại — chỉ chạy notebook này 1 lần duy nhất.

**Output cuối có gì**:
- `artifact/` — toàn bộ mesh `piece_X.obj` đã decompress, ~3–5GB
- `data_split/` — train/val splits (`artifact.train.txt`, `artifact.val.txt`)
- `sample_manifest.json` — log fractures đã chọn (cho reproducibility)

Khi attach dataset này vào notebook training, mọi thứ sẵn sàng để viết Dataset class + train DiffusionNet.